<a href="https://colab.research.google.com/github/anilpomar/Full-Stack-Gen-AI-BootCamp-KrishNaik-/blob/main/Assignment_Stage1_Stage2_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install stable torch (2.5.1 is more stable than 2.4.0)
!pip install torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.51.0
!pip install trl==0.23.0
!pip install peft accelerate bitsandbytes xformers

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl (664.8 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_curand_

  Using cached transformers-4.51.0-py3-none-any.whl.metadata (38 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.51.0-py3-none-any.whl (10.4 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.22.0
    Uninstalling huggingface_hub-1.22.0:
      Successfully uninstalled huggingface_hub-1.22.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-

  Using cached trl-0.23.0-py3-none-any.whl.metadata (11 kB)
  Using cached transformers-5.13.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.22.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached trl-0.23.0-py3-none-any.whl (564 kB)
Using cached transformers-5.13.0-py3-none-any.whl (11.5 MB)
Using cached huggingface_hub-1.22.0-py3-none-any.whl (765 kB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.4
    Uninstalling tokenizers-0.21.4:
      Successfully uninstalled tokenizers-0.21.4
  Attempting uninstall: transformers
    Found existing inst

In [ ]:
# Install unsloth from PyPI (NOT git) - more stable
!pip install -q unsloth transformers==5.5.0 trl==0.24.0 peft accelerate bitsandbytes
!pip install PyMuPDF

In [ ]:
# -------------------------
# 2. Imports
# -------------------------

import torch
import unsloth

print(f"Torch: {torch.__version__}")  # Should be 2.5.1
print(f"Unsloth: {unsloth.__version__}")

import os
import re
import gc
import time
import json
import unicodedata
import warnings
from typing import List, Dict, Any
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

import torch
import fitz  # PyMuPDF
from datasets import Dataset, load_dataset


from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
    print("DPO patch applied.")
except Exception as e:
    print("DPO patch skipped:", repr(e))

from trl import DPOTrainer, DPOConfig

assert torch.cuda.is_available(), "GPU not found. In Colab: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
Torch: 2.10.0+cu128
Unsloth: 2026.7.1
DPO patch applied.
GPU: Tesla T4


In [ ]:
from dataclasses import dataclass
# -------------------------
# 3. File paths
# -------------------------

non_instruction_data_path = "/content/non_instruction_dataset.pdf"
instruction_data_path = "/content/instruction_dataset.jsonl"
preference_data_path = "/content/preference_dataset.jsonl"

for path in [non_instruction_data_path, instruction_data_path, preference_data_path]:
  if not os.path.exists(path):
      raise FileNotFoundError(f"File not found: {path}. Please upload this file to Colab.")

# -------------------------
# 4. Configuration
# -------------------------
BASE_MODEL_NAME = "unsloth/tinyllama-bnb-4bit"

MAX_SEQ_LENGTH =2048 #512
SEED =3407 # was 42
MIN_CHARS_PER_PARAGRAPH = 80

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0

BATCH_SIZE = 2    #This is per_device_train_batch_size
GRAD_ACCUM_STEPS =8  #was 2 #16  This is gradient_accumulation_steps
WARMUP_STEPS = 5 #10
LOGGING_STEPS = 1

# Classroom demo steps. Increase for serious training.
STAGE1_MAX_STEPS = 15
STAGE2_MAX_STEPS = 80
STAGE3_MAX_STEPS = 20

STAGE1_LR = 5e-5       #This is learning_rate
STAGE2_LR = 2e-5        #This is learning_rate
STAGE3_LR = 5e-5       # This is learning_rate

DPO_BETA = 0.1

OUTPUT_ROOT = "/content/unsloth_pharma_merge_reload_outputs"

non_instruction_ADAPTER_DIR = f"{OUTPUT_ROOT}/non_instruction_adapter"
non_instruction_MERGED_DIR  = f"{OUTPUT_ROOT}/non_instruction_merged_model"

instruction_ADAPTER_DIR = f"{OUTPUT_ROOT}/instruction_adapter"
instruction_MERGED_DIR  = f"{OUTPUT_ROOT}/instruction_merged_model"

preference_ADAPTER_DIR = f"{OUTPUT_ROOT}/preference_dpo_adapter"
preference_FINAL_MERGED_DIR   = f"{OUTPUT_ROOT}/preference_dpo_final_merged_model"

for path in [
  OUTPUT_ROOT,
  non_instruction_ADAPTER_DIR,
  non_instruction_MERGED_DIR,
  instruction_ADAPTER_DIR,
  instruction_MERGED_DIR,
  preference_ADAPTER_DIR,
  preference_FINAL_MERGED_DIR,
]:
  os.makedirs(path, exist_ok=True)


In [ ]:
# ============================================================
# STAGE 1 DATA: PDF -> raw text dataset
# ============================================================

def build_pdf_dataset(pdf_path: str) -> Dataset:
    pages = extract_pdf_pages(pdf_path)
    records = []

    for page in pages:
        cleaned_text = clean_pdf_text(page["text"])

        for para_id, paragraph in enumerate(cleaned_text.split("\n\n"), start=1):
            paragraph = paragraph.strip()

            if len(paragraph) >= MIN_CHARS_PER_PARAGRAPH:
                records.append({
                    "text": paragraph,
                    "source_page": page["page"],
                    "paragraph_id": para_id,
                })

    if len(records) == 0:
        raise ValueError("No usable paragraph found. Try reducing MIN_CHARS_PER_PARAGRAPH.")

    print("PDF pages extracted:", len(pages))
    print("Paragraph records:", len(records))
    print("\nSample paragraph:\n", records[0]["text"][:700])

    return Dataset.from_list(records)

def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    pages = []

    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()
            if text:
                pages.append({
                    "page": page_number,
                    "text": text,
                })

    return pages

def clean_pdf_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by hyphen and newline
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Remove page-number-only lines
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Normalize spaces and paragraph breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    paragraphs = []

    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    return "\n\n".join(paragraphs)

In [ ]:
# -------------------------
# Helper functions
# -------------------------
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()

def load_unsloth_model_with_lora(model_name_or_path: str):
    """
    Loads a base or merged model in 4-bit and attaches a fresh LoRA adapter.
    This is used at each stage:
    - Stage 1 loads BASE_MODEL_NAME
    - Stage 2 loads STAGE1_MERGED_DIR
    - Stage 3 loads STAGE2_MERGED_DIR
    """

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ], #"embed_tokens", "lm_head"
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth", # was True,
        random_state=SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer

def build_instruction_prompt(instruction: str, input_text: str = "") -> str:
    instruction = str(instruction).strip()
    input_text = str(input_text).strip()

    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    return f"### Instruction:\n{instruction}\n\n### Response:\n"

def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train() #Model Training go here.
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result

def generate_answer(model, tokenizer, instruction: str, input_text: str = "", max_new_tokens: int = 150):
    FastLanguageModel.for_inference(model)

    prompt = build_instruction_prompt(instruction, input_text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            # temperature=0.7,
            # top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_tokens = inputs["input_ids"].shape[-1]
    generated_tokens = output[0][input_tokens:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str, stage_name: str):
    """
    Saves LoRA adapter separately and also saves a merged standalone model.
    The merged model becomes the starting point for the next stage.
    """

    print(f"\nSaving {stage_name} adapter...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"{stage_name} adapter saved to:", adapter_dir)

    print(f"\nMerging {stage_name} adapter with base model...")
    FastLanguageModel.for_training(model)

    model.save_pretrained_merged(
        merged_dir,
        tokenizer,
        save_method="merged_16bit",
    )

    print(f"{stage_name} merged model saved to:", merged_dir)

In [ ]:
non_instruction_dataset = build_pdf_dataset(non_instruction_data_path)

PDF pages extracted: 26
Paragraph records: 25

Sample paragraph:
 Exhibit 10.3 Manufacturing Agreement Between Antares Pharma, Inc. and AMAG Pharmaceuticals, Inc. MANUFACTURING AGREEMENT This Manufacturing Agreement ("Agreement") is made and entered into as of the 20th day of March, 2018 (the "Effective Date") by and between Antares Pharma, Inc., a Delaware corporation, with offices located at 100 Princeton South, Suite 300, Ewing, NJ 08628 ("Antares"), and AMAG Pharmaceuticals, Inc., a Delaware corporation, with a corporate address at 1100 Winter Street, Waltham, MA 02451 ("AMAG"). Antares and AMAG are sometimes referred to herein individually as a "Party" and collectively as the "Parties". Recitals WHEREAS, AMAG is engaged in discovering, developing and 


In [ ]:
# -------------------------
# Non Instruction FineTuning
# -------------------------

stage1_model, tokenizer = load_unsloth_model_with_lora(BASE_MODEL_NAME)

FastLanguageModel.for_training(stage1_model)

stage1_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage1_logs",

    max_steps=STAGE1_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE1_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=True,
    weight_decay = 0.01,
    max_grad_norm = 0.3,
    lr_scheduler_type = "cosine",
    seed=SEED,
    warmup_ratio = 0.1,
    # embedding_learning_rate=  STAGE1_EMBR
)

stage1_trainer = SFTTrainer(
    model=stage1_model,
    processing_class=tokenizer,
    train_dataset=non_instruction_dataset,
    args=stage1_config,
)

train_and_measure(stage1_trainer, "STAGE 1 - NON-INSTRUCTION  TRAINING")


save_adapter_and_merge(
    model=stage1_model,
    tokenizer=tokenizer,
    adapter_dir=non_instruction_ADAPTER_DIR,
    merged_dir=non_instruction_MERGED_DIR,
    stage_name="Stage 1",
)

# #Plot Loss Graph
# losses = [h["loss"] for h in stage1_trainer.state.log_history if "loss" in h]
# losses = np.array(losses)
# k = 5
# roll = np.convolve(losses, np.ones(k)/k, mode="valid")
# roll_x = np.arange(k//2, k//2 + len(roll))   # center the window

# plt.figure(figsize=(9, 4))
# plt.plot(losses, ":", color="#898781", marker=".", label="raw loss")
# plt.plot(roll_x, roll, "-", color="#2a78d6", linewidth=2.5, label="rolling avg (5)")
# plt.axhline(losses.mean(), color="#b4b2a9", linestyle="--", label=f"mean {losses.mean():.3f}")
# plt.xlabel("step"); plt.ylabel("training loss"); plt.legend()
# plt.title(f"range={losses.max()-losses.min():.3f}  drift={roll[-1]-roll[0]:+.3f}")
# plt.show()

del stage1_trainer
del stage1_model
clear_gpu_memory()


==((====))==  Unsloth 2026.7.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Unsloth: Will load unsloth/tinyllama-bnb-4bit as a legacy tokenizer.
Unsloth 2026.7.1 patched 22 layers with 22 QKV layers, 22 O layers and 22 MLP layers.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/25 [00:00<?, ? examples/s]

...

Stage 1 adapter saved to: /content/unsloth_pharma_merge_reload_outputs/non_instruction_adapter

Merging Stage 1 adapter with base model...


Unsloth: Restored added_tokens_decoder metadata in /content/unsloth_pharma_merge_reload_outputs/non_instruction_merged_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:35<00:00, 35.19s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_pharma_merge_reload_outputs/non_instruction_merged_model`
Stage 1 merged model saved to: /content/unsloth_pharma_merge_reload_outputs/non_instruction_merged_model


***Instruction Fine tuning (SFT)***


In [ ]:
# from datasets import load_dataset

# raw = load_dataset("json", data_files=instruction_data_path, split="train")

# ALPACA_TEMPLATE = (
#     "Below is an instruction that describes a task. "
#     "Write a response that appropriately completes the request.\n\n"
#     "### Instruction:\n{instruction}\n\n### Response:\n{response}"
# )
# RESPONSE_MARKER = "### Response:\n"

# def format_and_mask(example):
#     instr = example["instruction"]
#     inp = example.get("input", "")
#     if inp and inp.strip():
#         instr = f"{instr}\n\n### Input:\n{inp}"
#     text = ALPACA_TEMPLATE.format(instruction=instr, response=example["response"])
#     text += tokenizer.eos_token

#     enc = tokenizer(text, truncation=True, max_length=MAX_SEQ_LENGTH,
#                     add_special_tokens=False)
#     labels = list(enc["input_ids"])

#     # mask everything up to and including "### Response:\n"
#     split = text.find(RESPONSE_MARKER) + len(RESPONSE_MARKER)
#     n_prompt = len(tokenizer(text[:split], add_special_tokens=False)["input_ids"])
#     for i in range(min(n_prompt, len(labels))):
#         labels[i] = -100
#     enc["labels"] = labels
#     return enc

# stage2_dataset = raw.map(format_and_mask, remove_columns=raw.column_names)

# # verify the mask: should print ONLY the answer + </s>, not the instruction
# ex = stage2_dataset[0]
# trained = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]
# print("TRAINED-ON:", repr(tokenizer.decode(trained)))

In [ ]:
# from datasets import load_dataset

# # Load raw jsonl
# raw = load_dataset("json", data_files=instruction_data_path, split="train")

# # Alpaca prompt template — response marker MUST match train_on_responses_only
# ALPACA_TEMPLATE = (
#     "Below is an instruction that describes a task. "
#     "Write a response that appropriately completes the request.\n\n"
#     "### Instruction:\n{instruction}\n\n### Response:\n{response}"
# )

# def format_alpaca(example):
#     instr = example["instruction"]
#     inp = example.get("input", "")
#     # fold non-empty input into the instruction block
#     if inp and inp.strip():
#         instr = f"{instr}\n\n### Input:\n{inp}"
#     text = ALPACA_TEMPLATE.format(instruction=instr, response=example["response"])
#     text += tokenizer.eos_token          # critical: teaches the model to stop
#     return {"text": text}

# stage2_dataset = raw.map(format_alpaca, remove_columns=raw.column_names)

# # sanity check the formatted text
# print(repr(stage2_dataset[0]["text"]))

In [ ]:
# print("\n==============================")
# print("STAGE 2: INSTRUCTION Fine Tuning DATA")
# print("==============================")

# instruction_dataset = load_dataset(
#     "json",
#     data_files=instruction_data_path,
#     split="train",
# )


# required_instruction_cols = {"instruction", "response"}
# missing_cols = required_instruction_cols - set(instruction_dataset.column_names)

# if missing_cols:
#     raise ValueError(f"Instruction dataset missing columns: {missing_cols}")


# def format_instruction_record(example):
#     instruction = example.get("instruction", "")
#     input_text = example.get("input", "")
#     output = example.get("output", "")

#     text = build_instruction_prompt(instruction, input_text) + str(output).strip()
#     return {"text": text}

# stage2_dataset = instruction_dataset.map(format_instruction_record)

# print("Instruction rows:", len(stage2_dataset))
# print("\nSample instruction text:\n", stage2_dataset[0]["text"][:900])
# print(repr(stage2_dataset[0]["text"]))

In [ ]:
# ============================================================
# STAGE 2: Load Stage 1(Non Instruction) merged model -> Instruction Fine Tuning(SFT)
# ============================================================

stage2_model, tokenizer = load_unsloth_model_with_lora(non_instruction_MERGED_DIR)

#First Execute Above Line (load_unsloth_model_with_lora)
#Load Dataset now then execute below lines
#COmment line of code (load_unsloth_model_with_lora) and Uncomment below line

from datasets import load_dataset

raw = load_dataset("json", data_files=instruction_data_path, split="train")

ALPACA_TEMPLATE = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:\n{response}"
)
RESPONSE_MARKER = "### Response:\n"

def format_and_mask(example):
    instr = example["instruction"]
    inp = example.get("input", "")
    if inp and inp.strip():
        instr = f"{instr}\n\n### Input:\n{inp}"
    text = ALPACA_TEMPLATE.format(instruction=instr, response=example["response"])
    text += tokenizer.eos_token

    enc = tokenizer(text, truncation=True, max_length=MAX_SEQ_LENGTH,
                    add_special_tokens=False)
    labels = list(enc["input_ids"])

    # mask everything up to and including "### Response:\n"
    split = text.find(RESPONSE_MARKER) + len(RESPONSE_MARKER)
    n_prompt = len(tokenizer(text[:split], add_special_tokens=False)["input_ids"])
    for i in range(min(n_prompt, len(labels))):
        labels[i] = -100
    enc["labels"] = labels
    return enc

stage2_dataset = raw.map(format_and_mask, remove_columns=raw.column_names)

# verify the mask: should print ONLY the answer + </s>, not the instruction
ex = stage2_dataset[0]
trained = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]
print("TRAINED-ON:", repr(tokenizer.decode(trained)))


# from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(stage2_model)
tokenizer.padding_side = "right"

stage2_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage2_logs",

    max_steps=STAGE2_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE2_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,

    seed=SEED,
)

stage2_trainer = SFTTrainer(
    model=stage2_model,
    processing_class=tokenizer,
    train_dataset=stage2_dataset,
    args=stage2_config,
)

# stage2_trainer = train_on_responses_only(
#     stage2_trainer,
#     instruction_part="### Instruction:\n",
#     response_part="### Response:",     # removed the \n
# )

train_and_measure(stage2_trainer, "STAGE 2 - INSTRUCTION FINE-TUNING")

print("\nStage 2 test answer:")
print(generate_answer(stage2_model, tokenizer, "What is the Drug that is the subject of the Antares–AMAG Manufacturing Agreement.", max_new_tokens=120))


# stage2_model.print_trainable_parameters()
save_adapter_and_merge(
    model=stage2_model,
    tokenizer=tokenizer,
    adapter_dir=instruction_ADAPTER_DIR,
    merged_dir=instruction_MERGED_DIR,
    stage_name="SFT-Instruction Fine Tuning",
)

del stage2_trainer
del stage2_model
clear_gpu_memory()

==((====))==  Unsloth 2026.7.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Unsloth: Will load /content/unsloth_pharma_merge_reload_outputs/non_instruction_merged_model as a legacy tokenizer.


trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338
TRAINED-ON: 'The Drug is 17-alpha hydroxyprogesterone caproate.</s>'


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 12 | Total steps = 80
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


...

Under the Antares–AMAG Manufacturing Agreement, AMAG manufactures and sells 370 Units of Orencia to Amgen Inc., pursuant to an amended Manufacturing Supply Agreement ("MSA") (formerly known as the MSA). The MSM allows Amgen to substitute any drug containing the manufactured dose of Orencia that AMAG actually manufactures for delivery upon receipt, but Amgen must notify AMAG in writing thirty days prior to taking delivery of the alternate drug; and

Saving SFT-Instruction Fine Tuning adapter...
SFT-Instruction Fine Tuning adapter saved to: /content/unsloth_pharma_merge_reload_outputs/instruction_adapter

Merging SFT-Instruction Fine Tuning adapter with base model...
Detected local model directory: /content/unsloth_pharma_merge_reload_outputs/non_instruction_merged_model
Copied tokenizer.model from local model directory


Unsloth: Restored added_tokens_decoder metadata in /content/unsloth_pharma_merge_reload_outputs/instruction_merged_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:40<00:00, 40.25s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_pharma_merge_reload_outputs/instruction_merged_model`
SFT-Instruction Fine Tuning merged model saved to: /content/unsloth_pharma_merge_reload_outputs/instruction_merged_model
